# Week 6 — Used Car Price Predictor

**Theme:** Data representation — categorical encoding & feature engineering

Models only understand numbers. Real-world data is full of text categories
("SUV", "diesel", "manual") and raw values that hide the pattern a model
actually needs (a purchase *year* is less useful than the car's *age*). This
week we build a used-car dataset and improve how we represent it in four
steps, watching the prediction error drop at each one.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

## 1. Build a synthetic used-car dataset

We generate 300 used car listings with a known (but noisy) pricing rule, so we
can see clearly how much each data-representation trick helps.

In [ ]:
rng = np.random.default_rng(7)
n = 300

brands = rng.choice(["Hyundai", "Kia", "Toyota", "BMW"], size=n, p=[0.35, 0.3, 0.2, 0.15])
fuel_types = rng.choice(["gasoline", "diesel", "electric"], size=n, p=[0.55, 0.3, 0.15])
transmissions = rng.choice(["automatic", "manual"], size=n, p=[0.8, 0.2])
purchase_year = rng.integers(2010, 2024, size=n)
mileage_km = rng.integers(5_000, 180_000, size=n)

brand_premium = {"Hyundai": 0, "Kia": 0, "Toyota": 3_000, "BMW": 12_000}
fuel_premium = {"gasoline": 0, "diesel": 1_000, "electric": 6_000}

age = 2024 - purchase_year
base_price = 28_000
price = (
    base_price
    - age * 1_400                 # older car -> cheaper
    - mileage_km * 0.05           # more mileage -> cheaper
    + np.array([brand_premium[b] for b in brands])
    + np.array([fuel_premium[f] for f in fuel_types])
    + rng.normal(0, 1_500, size=n)  # noise: negotiation, condition, luck
)
price = np.clip(price, 2_000, None)

cars = pd.DataFrame({
    "brand": brands,
    "fuel_type": fuel_types,
    "transmission": transmissions,
    "purchase_year": purchase_year,
    "mileage_km": mileage_km,
    "price": price.round(0),
})
cars.head()

## Step 1: Numbers-only baseline (ignore the categories)

The naive approach: drop every text column and use only the numeric ones.

In [ ]:
def evaluate(X, y, label):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)
    model = LinearRegression().fit(Xtr, ytr)
    mae = mean_absolute_error(yte, model.predict(Xte))
    print(f"{label:35s} MAE = ${mae:,.0f}")
    return mae

results = {}
X1 = cars[["purchase_year", "mileage_km"]]
results["1. numeric only"] = evaluate(X1, cars["price"], "1. Numeric columns only")

## Step 2: One-hot encode the categorical columns

`pd.get_dummies` turns each category into its own 0/1 column, so "brand" (a
word) becomes 4 numeric columns the model can use — without implying any order
between brands.

In [ ]:
categorical_encoded = pd.get_dummies(cars[["brand", "fuel_type", "transmission"]])
categorical_encoded.head()

In [ ]:
X2 = pd.concat([cars[["purchase_year", "mileage_km"]], categorical_encoded], axis=1)
results["2. + one-hot categories"] = evaluate(X2, cars["price"], "2. + one-hot encoded categories")

## Step 3: Feature engineering — `age` instead of `purchase_year`

A linear model can only learn "the *bigger* this number, the *more/less* the
price changes." `purchase_year` (e.g. 2010 vs. 2023) works, but `age` (car's
age in years) is a more natural, directly-meaningful feature for the same
underlying relationship. Feature engineering means *reshaping* raw values into
a form that's easier for the model to use.

In [ ]:
cars["age"] = 2024 - cars["purchase_year"]

X3 = pd.concat([cars[["age", "mileage_km"]], categorical_encoded], axis=1)
results["3. + engineered 'age' feature"] = evaluate(X3, cars["price"], "3. + engineered 'age' feature")

## Step 4: Scale the numeric features

`mileage_km` ranges up to 180,000 while `age` maxes out around 14 — very
different scales. Some models (and especially neural networks, coming in
Week 9) train better when every feature is standardized to a similar range.

In [ ]:
numeric_cols = ["age", "mileage_km"]
scaler = StandardScaler()
X4 = X3.copy()
X4[numeric_cols] = scaler.fit_transform(X4[numeric_cols])

results["4. + scaled numeric features"] = evaluate(X4, cars["price"], "4. + scaled numeric features")

## Compare all four representations

In [ ]:
summary = pd.Series(results)
print(summary)

plt.figure(figsize=(7, 4))
summary.plot(kind="barh", color="steelblue")
plt.title("Prediction Error (MAE, lower = better) by Data Representation")
plt.xlabel("Mean Absolute Error ($)")
plt.gca().invert_yaxis()
plt.show()

## Try it yourself

1. **Label encoding instead of one-hot.** Try
   `cars["brand"].map({"Hyundai": 0, "Kia": 1, "Toyota": 2, "BMW": 3})` as a
   single numeric column instead of one-hot encoding. Does MAE get worse?
   Why might a linear model be misled by pretending brand is an ordered number?
2. **Add another engineered feature.** Create `mileage_per_year = mileage_km / age`
   (careful with `age == 0`) and add it to step 4 — does it help?
3. **Which single feature matters most?** Look at the model's `.coef_` after
   fitting on `X4` — which feature has the largest (absolute) coefficient?
4. **Standardizing the target?** We didn't scale `price` (the target) — why is
   that fine for regression, unlike the input features?

---
## 🎯 캡스톤: 우리 학교 중고거래 적정가 산출기

캠퍼스 중고거래(전공서적, 전자기기, 의류 등)의 가상 거래 기록 200건을 드립니다. 위에서 배운 **원-핫 인코딩 + feature engineering + 선형회귀**를 그대로 적용해서, 여러분이 팔고 싶은 물건의 적정가를 예측하는 모델을 직접 만들어보세요.

**확장 아이디어:** 실제 학교 커뮤니티/중고거래 게시판에서 거래 완료 글 수십 건(카테고리, 상태, 원가, 사용기간, 거래가)을 모아 `items_df`를 바꾸면, 진짜 "적정가 계산기"가 됩니다.

In [ ]:
# 더미 데이터 생성 (실행만 하면 됩니다)
rng = np.random.default_rng(5)
n_items = 200

categories = ["전공서적", "교양서적", "전자기기", "의류", "기타"]
conditions = ["새것같음", "양호", "사용감있음", "많이닳음"]

price_range = {"전공서적": (30000, 90000), "교양서적": (10000, 30000),
               "전자기기": (200000, 1500000), "의류": (20000, 150000), "기타": (5000, 50000)}
decay_rate = {"전공서적": 0.005, "교양서적": 0.01, "전자기기": 0.03, "의류": 0.02, "기타": 0.015}
condition_factor = {"새것같음": 0.95, "양호": 0.8, "사용감있음": 0.65, "많이닳음": 0.45}

cat_choices = rng.choice(categories, n_items)
cond_choices = rng.choice(conditions, n_items)
age_months = rng.integers(1, 48, n_items)
original_price = np.array([rng.uniform(*price_range[c]) for c in cat_choices])

decay = np.array([max(0.15, 1 - decay_rate[c] * m) for c, m in zip(cat_choices, age_months)])
cond_mult = np.array([condition_factor[c] for c in cond_choices])
sold_price = original_price * decay * cond_mult * rng.normal(1.0, 0.08, n_items)

items_df = pd.DataFrame({
    "category": cat_choices,
    "condition": cond_choices,
    "age_months": age_months,
    "original_price": original_price.round(0),
    "sold_price": sold_price.round(0),
})
items_df.head()

### 여러분의 과제

1. `category`, `condition`을 `pd.get_dummies`로 원-핫 인코딩하세요.
2. `age_months`, `original_price`와 원-핫 인코딩된 컬럼들을 합쳐 입력(X)을 만들고, `sold_price`를 정답(y)으로 하여 `LinearRegression`을 학습시키세요. (위 본문의 `evaluate()` 함수를 그대로 재사용해도 됩니다.)
3. 여러분이 실제로 팔고 싶은(또는 사고 싶은) 물건 하나를 골라 아래 `my_item`에 정보를 채우고, 학습된 모델로 적정가를 예측해보세요. **주의:** 예측할 때 입력 데이터의 컬럼 순서/이름이 학습 때와 정확히 같아야 합니다.

In [ ]:
# TODO 1: category, condition을 원-핫 인코딩하세요.


# TODO 2: age_months, original_price + 원-핫 컬럼으로 입력(X)을 구성하고 LinearRegression을 학습시키세요.


# TODO 3: 나의 물건 정보를 입력하고 적정가를 예측해보세요.
my_item = {
    "category": None,        # 예: "전자기기"
    "condition": None,       # 예: "양호"
    "age_months": None,      # 예: 10
    "original_price": None,  # 예: 800000
}